<a href="https://colab.research.google.com/github/CSUC/RDR-scripts/blob/main/move_dataset/move_dataset_script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Script to move datasets between dataverses
### OBSERVATION:
This script is available in the following GitHub repository: <a href='https://github.com/CSUC/RDR-scripts/tree/main/related_publication_check' target='_blank'>RDR-scripts</a>. </p> If you have questions or doubts about the code, please contact rdr-contacte@csuc.cat.
### SCRIPT OBJECTIVE:
The main objective of this script is to move a dataset from one instance to another.
## REQUISITS DE L'SCRIPT
To move a dataset between instances through the API, you need to:

- have the Publisher role in the instance / sub-instance where the dataset is currently located
- have the Publisher role in the instance / sub-instance where you want to move the dataset

In [ ]:
# @title Install or Update Libraries (Click the Run button &#x25B6; )
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

def install_packages(b):
    """
    Function to install or update required Python packages.

    Args:
    b (widget): Button widget that triggers the installation process.

    Returns:
    None
    """
    clear_output(wait=True)
    !pip install --upgrade pip -q  # Upgrade pip silently
    !pip install pyDataverse -q    # Install or update pyDataverse silently
    !pip install requests -q       # Install or update requests silently
    print("Libraries downloaded or updated successfully.")

# Displaying installation message
display(HTML("<p style='font-size:14px;'><b>Click the button below to install the required libraries.</b></p>"))

# Creating installation button
install_button = widgets.Button(description='Install Libraries')
install_button.on_click(install_packages)

# Displaying the installation button
display(install_button)


In [ ]:
# @title Move Dataset by DOI (doi:10.34810/dataXXX), Token, and Target Dataverse Alias
# Click the Run button ▶

import requests
from pyDataverse.api import NativeApi


def move_dataset(api_token, persistent_id, alias, force_move=False):
    """
    Move a dataset to a different Dataverse collection.

    Args:
        api_token (str): API token for authentication.
        persistent_id (str): Persistent identifier, for example "doi:10.34810/DATA1939".
        alias (str): Alias of the target Dataverse collection.
        force_move (bool): If True, uses forceMove=true to remove incompatible
                           guestbook or Dataverse collection links if needed.

    Returns:
        None
    """

    server_url = "https://dataverse.csuc.cat"

    headers = {
        "X-Dataverse-key": api_token,
        "Accept": "application/json",
        "Connection": "close",
        "User-Agent": "Dataverse dataset mover",
    }

    # Get dataset information using the DOI / persistent ID
    api = NativeApi(server_url, api_token)

    try:
        response = api.get_dataset(persistent_id)
    except Exception as e:
        print("Could not retrieve dataset.")
        print(e)
        return

    if response.status_code != 200:
        print("Could not get dataset ID.")
        return

    try:
        dataset_id = response.json()["data"]["id"]
    except KeyError:
        print("Dataset ID not found in response.")
        return

    # Move the dataset
    move_url = f"{server_url}/api/datasets/{dataset_id}/move/{alias}"

    params = {}
    if force_move:
        params["forceMove"] = "true"

    try:
        move_response = requests.post(
            move_url,
            headers=headers,
            params=params,
            timeout=60,
        )
    except requests.exceptions.RequestException as e:
        print("Move request failed.")
        print(e)
        return

    if move_response.status_code == 200:
        print("Dataset move operation successful.")
    else:
        print("Dataset move operation failed.")


# Provide input values: DOI, TOKEN and ALIAS
identifier = "doi:10.34810/data"  # @param {type:"string"}
token = ""        # @param {type:"string"}
alias = ""            # @param {type:"string"}

# Move the dataset with forceMove=true
move_dataset(token, identifier, alias, force_move=True)